# UR10 Offline Evaluation + Rollout Collection (PPO)
This notebook loads a trained PPO policy (normalizer + policy + value params), **reconstructs the environment and network setup exactly like training**, then runs **offline rollouts** and **collects rewards + transitions**.

**Inputs you need locally (offline):**
- `params.msgpack` (from W&B artifact or your local checkpoint)
- (Recommended) `run_config.json` saved alongside the params so the notebook can rebuild the same `network_factory` kwargs and PPO config overrides.

If you don't have `run_config.json`, the notebook still runs, but it may not match training architecture if your run used non-default network sizes.


In [ ]:
# --- Standard libs ---
import os, json
import numpy as np

# --- JAX/Flax/Brax ---
import jax
import jax.numpy as jnp
from flax import serialization

from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics

# --- Mujoco Playground ---
from mujoco_playground.config import manipulation_params
from mujoco_playground import registry, wrapper

print("✓ Imports OK")

## 1) Paths and run metadata
Set the paths to your offline `params.msgpack` and (optionally) `run_config.json`.


In [ ]:
# --- EDIT THESE PATHS ---
PARAMS_PATH = "evaluation/downloaded_policies/params.msgpack"
CFG_PATH    = "evaluation/downloaded_policies/run_config.json"  # recommended if available

# Environment name you trained/evaluate on
ENV_NAME = "UR10PickCube"

# Rollout settings
NUM_ROLLOUTS = 50
SEED_BASE = 100

# Episode length: if you loaded CFG, we will auto-pull the training episode_length when possible
FALLBACK_EPISODE_LENGTH = 150

# Save dataset toggle
SAVE_DATASET = True
DATASET_OUT_PATH = "evaluation/graphs/offline_rollouts_dataset.npz"

# --- Load files ---
assert os.path.exists(PARAMS_PATH), f"PARAMS_PATH not found: {PARAMS_PATH}"

with open(PARAMS_PATH, "rb") as f:
    params_bytes = f.read()

cfg = {}
if os.path.exists(CFG_PATH):
    with open(CFG_PATH, "r") as f:
        cfg = json.load(f)

print("Loaded params bytes:", len(params_bytes))
print("Loaded cfg:", bool(cfg), "| keys sample:", list(cfg.keys())[:15])

## 2) Import training helpers from `UR10_ppo.py`
We reuse the same override logic as training:
- `_extract_ppo_overrides`
- `apply_validated_overrides`
- `cast_to_schema`

This ensures `ppo_params_overwrite` matches what training passed into `ppo.train`.


In [ ]:
# Try importing UR10_ppo.py from common locations.
# If your UR10_ppo.py is in a different folder, add it to sys.path below.
import sys

# Common in this environment: user uploaded to /mnt/data
if "/mnt/data" not in sys.path:
    sys.path.append("/mnt/data")

try:
    from UR10_ppo import _extract_ppo_overrides, apply_validated_overrides, cast_to_schema
    print("✓ Imported helpers from UR10_ppo.py")
except Exception as e:
    raise ImportError(
        "Could not import UR10_ppo.py helpers. Ensure UR10_ppo.py is available in your working directory "
        "or /mnt/data, or append its folder to sys.path."
    ) from e

## 3) Rebuild PPO config + network_factory exactly like training
This mirrors the key logic in `UR10_ppo.py`:
- start from `manipulation_params.brax_ppo_config(ENV_NAME)`
- apply validated overrides extracted from `cfg` (if provided)
- resolve `network_factory` kwargs


In [ ]:
import functools

# --- Base config from mujoco_playground (same as training) ---
base = manipulation_params.brax_ppo_config(ENV_NAME)
try:
    base_dict = base.to_dict()
except AttributeError:
    base_dict = dict(base)

ppo_params = dict(base_dict)

# --- Apply cfg overrides like training ---
overrides = _extract_ppo_overrides(cfg) if cfg else {}
ppo_params = apply_validated_overrides(ppo_params, overrides, strict=True)

# Cast to schema (same behavior as training)
schema = dict(base_dict)
schema.pop("network_factory", None)
ppo_params_overwrite = cast_to_schema(ppo_params, schema)

# Remove keys training passes explicitly / not accepted by ppo.train
for k in ["network_factory", "seed", "init_keyframe"]:
    ppo_params_overwrite.pop(k, None)

# --- Resolve network_factory kwargs exactly like training ---
nf_params = dict(ppo_params.get("network_factory") or {})
if isinstance(nf_params.get("policy_hidden_layer_sizes"), list):
    nf_params["policy_hidden_layer_sizes"] = tuple(nf_params["policy_hidden_layer_sizes"])
if isinstance(nf_params.get("value_hidden_layer_sizes"), list):
    nf_params["value_hidden_layer_sizes"] = tuple(nf_params["value_hidden_layer_sizes"])

network_factory = ppo_networks.make_ppo_networks
if isinstance(nf_params, dict) and len(nf_params) > 0:
    network_factory = functools.partial(ppo_networks.make_ppo_networks, **nf_params)

print("Resolved network_factory kwargs:", nf_params)
print("ppo_params_overwrite keys:", sorted(list(ppo_params_overwrite.keys()))[:30], "...")

## 4) Build + wrap env the same way as training
Training used `wrapper.wrap_for_brax_training`.  
We apply it directly, passing `episode_length` and `action_repeat` if present.


In [ ]:
# --- Load base env ---
env = registry.load(ENV_NAME)

# --- Wrap env like training ---
wrap_kwargs = {}
for k in ["episode_length", "action_repeat"]:
    if k in ppo_params_overwrite:
        wrap_kwargs[k] = int(ppo_params_overwrite[k])

env = wrapper.wrap_for_brax_training(env, **wrap_kwargs)

jit_reset = jax.jit(env.reset)
jit_step  = jax.jit(env.step)

obs_size = env.observation_size
action_size = env.action_size

print(f"Obs size={obs_size} | Action size={action_size}")
print("Wrap kwargs:", wrap_kwargs)

## 5) Build networks + restore params
We use the same `preprocess_observations_fn=running_statistics.normalize` setup as training and restore:
- normalizer params (`params["0"]`)
- policy params (`params["1"]`)
- value params (`params["2"]`)


In [ ]:
# --- Build PPO networks ---
normalize = running_statistics.normalize
ppo_network = network_factory(
    observation_size=obs_size,
    action_size=action_size,
    preprocess_observations_fn=normalize,
)

# --- Create dummy template for safe deserialization ---
rng = jax.random.PRNGKey(0)

dummy_normalizer_params = running_statistics.init_state(
    jax.ShapeDtypeStruct((obs_size,), jnp.float32)
)
dummy_policy_params = ppo_network.policy_network.init(rng)
dummy_value_params  = ppo_network.value_network.init(rng)

params_template = {"0": dummy_normalizer_params, "1": dummy_policy_params, "2": dummy_value_params}

# --- Restore ---
params = serialization.from_bytes(params_template, params_bytes)

normalizer_params = params["0"]
policy_params     = params["1"]
value_params      = params["2"]

print("✓ Params restored (normalizer/policy/value)")

## 6) Offline rollouts + reward collection
This runs deterministic actions (policy mean).  
It collects:
- `episode_returns` for each rollout
- `transitions`: (obs, action, reward, done, next_obs)

If you want stochastic sampling instead of mean actions, change `policy_action(...)`.


In [ ]:
def policy_action(obs):
    """Deterministic: use mean action output."""
    raw = ppo_network.policy_network.apply(normalizer_params, policy_params, obs)
    return raw[:action_size]

def run_rollouts(num_rollouts=50, episode_length=150, seed_base=100, collect_transitions=True):
    returns = []
    transitions = []

    for i in range(num_rollouts):
        rng = jax.random.PRNGKey(seed_base + i)
        rng, reset_rng = jax.random.split(rng)
        state = jit_reset(reset_rng)

        ep_return = 0.0

        for t in range(episode_length):
            obs = state.obs
            action = policy_action(obs)

            next_state = jit_step(state, action)

            r = float(next_state.reward)
            d = bool(next_state.done) if hasattr(next_state, "done") else False

            ep_return += r

            if collect_transitions:
                transitions.append({
                    "obs": np.array(obs),
                    "action": np.array(action),
                    "reward": r,
                    "done": d,
                    "next_obs": np.array(next_state.obs),
                })

            state = next_state
            if d:
                break

        returns.append(ep_return)

        if (i + 1) % 10 == 0:
            print(f"Completed {i+1}/{num_rollouts}")

    return np.array(returns), transitions

episode_length = int(ppo_params_overwrite.get("episode_length", FALLBACK_EPISODE_LENGTH))
episode_returns, transitions = run_rollouts(
    num_rollouts=NUM_ROLLOUTS,
    episode_length=episode_length,
    seed_base=SEED_BASE,
    collect_transitions=True
)

print("\n✓ Done")
print("Episode length used:", episode_length)
print("Mean return:", episode_returns.mean())
print("Std return:", episode_returns.std())
print("Min/Max:", episode_returns.min(), episode_returns.max())
print("Transitions collected:", len(transitions))

## 7) Save an offline dataset (optional)
Saves a compressed `.npz` with arrays: `obs, actions, rewards, dones, next_obs, episode_returns`, plus JSON metadata.


In [ ]:
if SAVE_DATASET:
    os.makedirs(os.path.dirname(DATASET_OUT_PATH), exist_ok=True)

    obs      = np.stack([tr["obs"] for tr in transitions], axis=0)
    actions  = np.stack([tr["action"] for tr in transitions], axis=0)
    rewards  = np.array([tr["reward"] for tr in transitions], dtype=np.float32)
    dones    = np.array([tr["done"] for tr in transitions], dtype=np.bool_)
    next_obs = np.stack([tr["next_obs"] for tr in transitions], axis=0)

    np.savez_compressed(
        DATASET_OUT_PATH,
        obs=obs,
        actions=actions,
        rewards=rewards,
        dones=dones,
        next_obs=next_obs,
        episode_returns=episode_returns,
        env_name=ENV_NAME,
        ppo_params_overwrite=json.dumps(ppo_params_overwrite),
        network_factory_kwargs=json.dumps(nf_params),
    )

    print("Saved dataset to:", DATASET_OUT_PATH)
    print("Shapes:",
          "obs", obs.shape,
          "| actions", actions.shape,
          "| rewards", rewards.shape,
          "| dones", dones.shape,
          "| next_obs", next_obs.shape)
else:
    print("SAVE_DATASET=False, skipping save.")

## 8) Quick plot of returns (optional)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(episode_returns)
plt.xlabel("Rollout")
plt.ylabel("Episode Return")
plt.title(f"Offline Evaluation Returns: {ENV_NAME}")
plt.show()